# Notebook 2: Simulación cosmológica ΛCDM

Este notebook muestra cómo correr una simulación cosmológica completa y analizar
el espectro de potencia de materia P(k).

**Conceptos cubiertos:**
- Condiciones iniciales Zel'dovich (1LPT)
- Integración con factores de drift/kick cosmológicos
- Espectro de potencia P(k) y comparación con teoría lineal
- Función de masa de halos (HMF)

**Requisitos:** `gadget-ng` compilado, numpy, scipy, matplotlib

In [ ]:
import json
import subprocess
import time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

PROJECT_ROOT = Path("..")
BINARY = PROJECT_ROOT / "target" / "release" / "gadget-ng"

assert BINARY.exists(), (
    f"Binario no encontrado: {BINARY}\n"
    "Ejecuta primero: cargo build --release -p gadget-ng-cli"
)
print(f"Binario: {BINARY.resolve()}")

## 1. Revisar la configuración cosmológica

El TOML `examples/cosmological.toml` define una simulación ΛCDM estándar
(parámetros Planck 2018): 512 partículas en una caja de 100 Mpc/h,
desde el redshift z=49 hasta z=0.

In [ ]:
config_path = PROJECT_ROOT / "examples" / "cosmological.toml"
print(config_path.read_text())

## 2. Correr la simulación cosmológica

In [ ]:
out_cosmo = PROJECT_ROOT / "runs" / "cosmo_nb02"

print("Corriendo simulación ΛCDM (512 partículas, z=49→0, 500 pasos)...")
t0 = time.time()

result = subprocess.run(
    [
        str(BINARY), "stepping",
        "--config", str(config_path),
        "--out", str(out_cosmo),
        "--snapshot",
    ],
    capture_output=True,
    text=True,
)

elapsed = time.time() - t0
print(f"Finalizado en {elapsed:.1f}s")

if result.returncode != 0:
    print("ERROR:", result.stderr[-1000:])
else:
    print("Archivos generados:")
    for f in sorted(out_cosmo.rglob("*")):
        print(f"  {f.relative_to(out_cosmo)}")

## 3. Análisis completo: FoF + P(k) + ξ(r) + c(M)

El subcomando `analyze` ejecuta en una sola pasada:
- **FoF** (Friends-of-Friends): catálogo de halos
- **P(k)**: espectro de potencia de materia
- **ξ(r)**: función de correlación de 2 puntos
- **c(M)**: relación concentración-masa (Ludlow+2016)

In [ ]:
out_analysis = out_cosmo / "analysis"
snap_dir = out_cosmo / "snapshot_final"

if not snap_dir.exists():
    # Buscar cualquier snapshot disponible
    snaps = list(out_cosmo.glob("snapshot*"))
    snap_dir = snaps[0] if snaps else out_cosmo

print(f"Analizando snapshot en: {snap_dir}")

result = subprocess.run(
    [
        str(BINARY), "analyze",
        "--snapshot", str(snap_dir),
        "--out", str(out_analysis),
    ],
    capture_output=True,
    text=True,
)

if result.returncode != 0:
    print("ERROR:", result.stderr[-500:])
else:
    print("Análisis completado. Archivos:")
    for f in sorted(out_analysis.rglob("*")):
        print(f"  {f.relative_to(out_analysis)}")

## 4. Leer y graficar los resultados de análisis

In [ ]:
results_file = out_analysis / "results.json"

if results_file.exists():
    with open(results_file) as f:
        results = json.load(f)
    print("Claves del results.json:", list(results.keys()))
else:
    print(f"No se encontró {results_file}")
    results = {}

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Simulación cosmológica ΛCDM — z=0", fontsize=14)

# ── Espectro de potencia P(k) ──────────────────────────────────────────────
ax = axes[0]
pk_data = results.get("power_spectrum", results.get("pk", {}))

if pk_data and "k" in pk_data and "power" in pk_data:
    k = np.array(pk_data["k"])
    pk = np.array(pk_data["power"])
    mask = pk > 0
    ax.loglog(k[mask], pk[mask], "o-", ms=4, lw=1.5, label="gadget-ng")

    # Referencia de la escala lineal (P(k) ∝ k^n_s con n_s=0.965)
    k_ref = k[mask]
    # Normalización aproximada al modo fundamental
    pk_lin = k_ref ** 0.965
    pk_lin *= pk[mask][0] / pk_lin[0]
    ax.loglog(k_ref, pk_lin, "--", alpha=0.6, label="P(k) ∝ k^{n_s} (lineal)", lw=1.5)
    ax.legend()
else:
    ax.text(0.5, 0.5, "P(k) no disponible en results.json",
            ha="center", va="center", transform=ax.transAxes, color="gray")

ax.set_xlabel(r"$k$ [$h$/Mpc]")
ax.set_ylabel(r"$P(k)$ [Mpc/$h$]³")
ax.set_title("Espectro de potencia P(k)")
ax.grid(True, alpha=0.3, which="both")

# ── Función de masa de halos (HMF) ─────────────────────────────────────────
ax = axes[1]
hmf_data = results.get("halo_mass_function", results.get("hmf", {}))
halos = results.get("halos", results.get("fof", {}))

# Intentar graficar distribución de masas de halos FoF
halo_masses = None
if isinstance(halos, list):
    halo_masses = [h.get("mass", h.get("m", 0)) for h in halos if h]
elif isinstance(halos, dict):
    halo_masses = halos.get("masses", halos.get("mass", []))

if halo_masses and len(halo_masses) > 1:
    masses = np.array(halo_masses)
    masses = masses[masses > 0]
    ax.hist(np.log10(masses), bins=20, edgecolor="black", alpha=0.8)
    ax.set_xlabel(r"$\log_{10}(M/M_\odot)$")
    ax.set_ylabel("Número de halos")
    ax.set_title(f"Distribución de masas FoF ({len(masses)} halos)")
    ax.grid(True, alpha=0.3)
else:
    ax.text(0.5, 0.5, "HMF / halos FoF no disponibles\nen results.json",
            ha="center", va="center", transform=ax.transAxes, color="gray")
    ax.set_title("Función de masa de halos (HMF)")

plt.tight_layout()
plt.show()

## 5. Visualización de la estructura de gran escala

In [ ]:
# Cargar partículas del snapshot final
def load_snapshot_jsonl(snapshot_dir: Path) -> dict | None:
    """Carga particles.jsonl y devuelve arrays numpy."""
    for fname in ["particles.jsonl", "particles.bin"]:
        pfile = snapshot_dir / fname
        if pfile.exists() and fname.endswith(".jsonl"):
            positions, velocities = [], []
            with open(pfile) as f:
                for line in f:
                    line = line.strip()
                    if not line:
                        continue
                    p = json.loads(line)
                    positions.append(p.get("pos", p.get("position", [0, 0, 0])))
                    velocities.append(p.get("vel", p.get("velocity", [0, 0, 0])))
            return {
                "pos": np.array(positions),
                "vel": np.array(velocities),
            }
    return None


snap_final = out_cosmo / "snapshot_final"
if not snap_final.exists():
    snaps = sorted(out_cosmo.glob("snapshot*"))
    snap_final = snaps[-1] if snaps else None

data = None
if snap_final:
    data = load_snapshot_jsonl(snap_final)

if data is not None:
    pos = data["pos"]
    vel = data["vel"]
    speed = np.linalg.norm(vel, axis=1)

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    fig.suptitle("Estructura de gran escala — ΛCDM z=0 (N=512)", fontsize=14)

    proj_pairs = [(0, 1, "XY"), (0, 2, "XZ"), (1, 2, "YZ")]
    for ax, (i, j, label) in zip(axes, proj_pairs):
        ax.hexbin(pos[:, i], pos[:, j], gridsize=40, cmap="inferno", mincnt=1)
        ax.set_xlabel(f"{label[0]} [Mpc/h]")
        ax.set_ylabel(f"{label[1]} [Mpc/h]")
        ax.set_title(f"Proyección {label}")

    plt.tight_layout()
    plt.show()
    print(f"\nVelocidad rms: {speed.mean():.4f} km/s")
    print(f"δ_rms (sobredensidad): {pos.std() / pos.mean():.4f}")
else:
    print("No se encontraron datos de partículas para visualizar.")

## 6. Postprocesar con el script de P(k)

El script `docs/scripts/postprocess_pk.py` genera una evolución temporal de P(k,z)
si la simulación guardó archivos de análisis in-situ.

In [ ]:
insitu_dir = out_cosmo / "insitu"

if insitu_dir.exists() and list(insitu_dir.glob("insitu_*.json")):
    pk_out = out_cosmo / "analysis" / "pk_evolution.json"
    result = subprocess.run(
        [
            sys.executable,
            str(PROJECT_ROOT / "docs" / "scripts" / "postprocess_pk.py"),
            "--insitu", str(insitu_dir),
            "--out", str(pk_out),
        ],
        capture_output=True,
        text=True,
    )
    print(result.stdout)
    if result.returncode != 0:
        print("STDERR:", result.stderr)
else:
    print("No hay archivos in-situ en esta corrida (necesita config con análisis in-situ activado).")
    print("Ver experiments/nbody/ para configs con in-situ habilitado.")

## Resumen

En este notebook aprendiste a:
- Interpretar una configuración TOML cosmológica
- Correr una simulación ΛCDM completa
- Ejecutar el pipeline de análisis (FoF + P(k) + ξ(r) + c(M))
- Visualizar la estructura de gran escala y el espectro de potencia

**Siguiente:** `03_herramientas_analisis.ipynb` — análisis detallado de halos y espectros.